#  AAS JSON →  Knowledge GraphConversion Pipeline

This notebook demonstrates the transformation of:
- an **ISO 63278 Asset Administration Shell (AAS) JSON** 
- via a **SPARQL CONSTRUCT mapping**  
- into  **OWL-based Knowledge Graph (KG)**  

Pipeline Steps:

1. Load AAS instance
2. Validate JSON against AAS Meta-Model schema
3. Convert AAS JSON -> RDF (using `py-aas-rdf`)
4. Apply SPARQL mapping → AAS RDF
5. Save output RDF



## Imports & Environment Setup
Make sure all the packages listed under the requirements.txt provided here are installed https://github.com/MI-FraunhoferIWM/AAS2KG-inspection-documents-of-steel-products

In [2]:
import json
import os
from rdflib import Graph, Namespace
from py_aas_rdf.models.submodel import Submodel
from pyoxigraph import RdfFormat, Store
import rdflib
from pathlib import Path
from pyshacl import validate


## Variable initialization 

In [3]:
# SPARQL construct related configuration. Do not modify unless you have to.
AAS_NS = Namespace("https://admin-shell.io/aas/3/0/")
BASE_URI = "https://example.org/aas/"
PLACEHOLDER = "PLACEHOLDER"

# Path to SPARQL construct mapping file (Specific to usecase)
INSPECTION_DOCUMENT_MAPPING_PATH = Path("../Input (AAS)/InspectionDocument_AAS2KG_mapping.sparql")
OUTPUT_DIR = Path("../Output (RDF)")
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "InspectionDocument_316_4401_alloy_PMDCore.ttl")

## Load AAS json data

In [11]:
# Path to pre downloaded AAS json file ( Replace the below code according to your need)
AAS_JSON_PATH = Path("../Input (AAS)/InspectionDocument_316_4401_alloy.json")
# Load AAS JSON
with open(AAS_JSON_PATH, "r", encoding="utf-8") as f:
    aas_json = json.load(f)

# Mapping file has PLACEHOLDER in the graph node names and you can simply replace it with any UNIQUE ID represnting your AAS file
UNIQUE_ID = "316-4401"

## Apply AAS2KG pipeline

In [12]:
       
# Run for each submodel in inspection document
for sm_json in aas_json.get("submodels", []):
    rdf_graph = Graph()
    rdf_graph.bind("aas", "https://admin-shell.io/aas/3/0/")
    
    # Validate submodel using py-aas-rdf library
    submodel = Submodel.model_validate(sm_json)
    # Convert submodel to RDF
    submodel.to_rdf(rdf_graph, base_uri="https://example.org/aas/",  id_strategy="base64-url-encode")
    ttl_output = rdf_graph.serialize(format="turtle")
    
    
    # Apply SPARQL construct for specific submodels 
    with open(INSPECTION_DOCUMENT_MAPPING_PATH, "r", encoding="utf-8") as f:
        mapping_query = f.read().replace(PLACEHOLDER, UNIQUE_ID)

        store = Store()

        store.load(ttl_output, format=RdfFormat.TURTLE)
        mapping_result = store.query(mapping_query)

        # Serialize the result from Oxigraph (returns bytes)
        mapped_bytes = mapping_result.serialize(format=RdfFormat.TURTLE)

        # --- Parse into rdflib.Graph for prefix binding ---
        g = rdflib.Graph()
        g.parse(data=mapped_bytes, format="turtle")

        g.bind("aas", "https://admin-shell.io/aas/3/0/")
        g.bind("ex", "http://www.example.org/#")
        g.bind("pmd", "https://w3id.org/pmd/co")
        g.bind("obo", "http://purl.obolibrary.org/obo")

        # Serialize with prefixes applied
        final_ttl = g.serialize(format="turtle")
        
        print("✓ Applied SPARQL mapping and namespace bindings.")
        print(final_ttl)
        # Write to file
        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            f.write(final_ttl)
        

✓ Applied SPARQL mapping and namespace bindings.
@prefix ex: <http://www.example.org/#> .
@prefix ns1: <http://purl.obolibrary.org/obo/> .
@prefix ns2: <https://w3id.org/pmd/co/> .
@prefix ns3: <https://qudt.org/schema/qudt/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:316-4401_chem_comp a ns2:PMD_0000551 ;
    ns1:RO_0000080 ex:316-4401_material ;
    ns2:PMD_0000004 ex:316-4401_chem_comp_spec .

ex:316-4401_mass_proportion_carbon a ns2:PMD_0020102 ;
    ns2:PMD_0000077 ex:316-4401_fraction_carbon ;
    ns2:PMD_0025999 ex:316-4401_some_carbon .

ex:316-4401_mass_proportion_chromium a ns2:PMD_0020102 ;
    ns2:PMD_0000077 ex:316-4401_fraction_chromium ;
    ns2:PMD_0025999 ex:316-4401_some_chromium .

ex:316-4401_mass_proportion_manganese a ns2:PMD_0020102 ;
    ns2:PMD_0000077 ex:316-4401_fraction_manganese ;
    ns2:PMD_0025999 ex:316-4401_some_manganese .

ex:316-4401_mass_proportion_molybdenum a ns2:PMD_0020102 ;
    ns2:PMD_0000077 ex:316-4401_fraction_molybdenum

## Execute SPARQL Query to fetch material properties

In [13]:
query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX n1: <http://purl.obolibrary.org/obo/>
PREFIX n2: <https://qudt.org/schema/qudt/>

SELECT DISTINCT ?BFO_0000040_8 ?RO_0000086_45 ?type_82 ?thing_81
                ?OBI_0001938_118 ?numericValue_155 ?unit_154
                ?label_193 ?type_191
WHERE {
    ?BFO_0000040_8 a n1:BFO_0000040 .
    ?BFO_0000040_8 n1:RO_0000086 ?RO_0000086_45 .
    ?RO_0000086_45 rdf:type ?type_82 .
    ?RO_0000086_45 n1:IAO_0000417 ?thing_81 .
    ?thing_81 n1:OBI_0001938 ?OBI_0001938_118 .
    ?OBI_0001938_118 n2:numericValue ?numericValue_155 .
    ?OBI_0001938_118 n2:unit ?unit_154 .
    ?BFO_0000040_8 rdfs:label ?label_193 .
    ?BFO_0000040_8 rdf:type ?type_191 .
}
LIMIT 200
"""

results = g.query(query)

for row in results:
    print("Material:", row.BFO_0000040_8)
    print("Role:", row.RO_0000086_45)
    print("Role type:", row.type_82)
    print("Thing:", row.thing_81)
    print("Measurement:", row.OBI_0001938_118)
    print("Value:", row.numericValue_155)
    print("Unit:", row.unit_154)
    print("Label:", row.label_193)
    print("Type:", row.type_191)
    print("------")

Material: http://www.example.org/#316-4401_material
Role: http://www.example.org/#316-4401_elongation_after_fracture
Role type: https://w3id.org/pmd/tto/TTO_0000033
Thing: http://www.example.org/#316-4401_elongation_after_fracture_scalar_value_specification
Measurement: http://www.example.org/#316-4401_elongation_after_fracture_value
Value: 60
Unit: https://qudt.org/schema/qudt/MegaPA
Label: 316/4401 – 2R-2BB – Cold Rolled Stainless Steel
Type: http://purl.obolibrary.org/obo/BFO_0000040
------
Material: http://www.example.org/#316-4401_material
Role: http://www.example.org/#316-4401_yield_strength
Role type: https://w3id.org/pmd/tto/TTO_0000009
Thing: http://www.example.org/#316-4401_yield_strength_scalar_value_specification
Measurement: http://www.example.org/#316-4401_yield_strength_value
Value: 276
Unit: https://qudt.org/schema/qudt/MegaPA
Label: 316/4401 – 2R-2BB – Cold Rolled Stainless Steel
Type: http://purl.obolibrary.org/obo/BFO_0000040
------
Material: http://www.example.org/#